In [2]:
import pandas as pd
import numpy as np

In [3]:
#\food_data_final.csv
df=pd.read_csv('./food_data_final.csv')


In [ ]:
input_ingre=['도토리묵', '상추', '당근', '양파']

['도토리묵', '상추', '당근', '양파']

In [6]:
df[df['rec1'].apply(lambda x: '도토리묵'in x and '상추' in x and '당근' in x and '양파' in x)]

,RCP_SNO,CKG_NM,CKG_MTRL_CN,rec1
9189,2603077,도토리묵무침,[재료] 도토리묵 1모| 상추 5장| 쑥갓 2줄기| 깻잎 4장| 양파 1/2개| 오...,"['도토리묵', '상추', '쑥갓', '깻잎', '양파', '오이', '당근', '..."
21121,6593198,도토리묵무침,[재료] 도토리묵 1모| 상추(or 깻잎) 4~5장| 양파 약간| 당근 약간 [양념...,"['도토리묵', '상추', '깻잎', '양파', '당근', '마늘']"
23396,6726120,도토리묵무침,[재료] 도토리묵| 상추| 깻잎| 양파| 오이| 당근| 고추 [양념] 고추가루 1T...,"['도토리묵', '상추', '깻잎', '양파', '오이', '당근', '고추', '..."
27172,6830788,도토리묵사발,[도토리묵 쑤기 재료] 도토리가루 1컵| 생수 5+1/2컵 [묵사발 재료] 도토리 ...,"['도토리묵', '쑤기', '도토리가루', '생수', '묵사발', '도토리', '오..."
35996,6841727,도토리묵무침,[재료] 도토리묵 1팩| 상추 10장| 양파 1개| 당근 1/5개| 청양고추 1개 ...,"['도토리묵', '상추', '양파', '당근', '청양고추', '마늘']"
...,...,...,...,...
202157,7035738,도토리묵무침,[재료] 도토리묵350g| 상추6장| 오이1/2개| 당근약간|...,"['도토리묵', '상추', '오이', '당근', '양파', '청양고추', '파', ..."
204653,7038364,도토리묵무침,[재료] 도토리묵1모(400g)| 깻잎10장( 또는 상추)| 오이1/3...,"['도토리묵', '깻잎', '상추', '오이', '당근', '양파', '마늘']"
205393,7039155,도토리묵,[재료] 도토리묵300g| 상추14장| 당근25g| 양파25g...,"['도토리묵', '상추', '당근', '양파', '마늘', '송송썬파']"
205676,7039450,도토리묵무침,[재료] 도토리묵1모| 상추5~6장| 오이1/2개| 깻잎5~6장...,"['도토리묵', '상추', '오이', '깻잎', '당근', '양파', '오이고추',..."


In [12]:
import pandas as pd
import json

def filter_recipes(input_ingre, df):
    """
    주어진 식재료 리스트(input_ingre)가 모두 포함된 레시피를 필터링하고
    결과를 JSON 형식으로 반환하는 함수
    """
    # 조건에 맞는 행 필터링
    filtered_df = df[df['rec1'].apply(lambda x: all(ingre in x for ingre in input_ingre))]

    # DataFrame을 JSON 형식으로 변환
    result_json = filtered_df.to_json(orient='records', force_ascii=False)
    return result_json


if __name__ == "__main__":
    # 예시 DataFrame

    # 앱에서 전달받은 재료 리스트 예시
    input_ingre = ['도토리묵', '상추', '당근', '양파']

    # 필터링된 결과 출력
    json_result = filter_recipes(input_ingre, df)
    print(len(json_result))


21805


In [21]:
import pandas as pd
import json
from itertools import combinations

def filter_recipes(input_ingre, df):
    """
    주어진 input_ingre 리스트로 레시피를 필터링하여 JSON으로 반환.
    1) 모든 재료가 포함된 행 우선
    2) 없으면 한 재료 빠진 조합들 중 포함된 행 반환
    3) 그래도 없으면 빈 JSON 리스트 반환
    """

    # ① 모든 재료 포함된 행
    exact_df = df[df['rec1'].apply(lambda x: all(ing in x for ing in input_ingre))]
    if not exact_df.empty:
        return exact_df.to_json(orient='records', force_ascii=False, indent=2)

    # ② 모든 재료 포함된 행이 없을 경우 → 한 재료 빠진 조합들
    n = len(input_ingre)
    partial_dfs = []
    for combo in combinations(input_ingre, n - 1):
        combo_list = list(combo)
        matched = df[df['rec1'].apply(lambda x: all(ing in x for ing in combo_list))]
        if not matched.empty:
            partial_dfs.append(matched)

    # 조합 중 하나라도 매칭된 것이 있다면 DataFrame 합치기
    if partial_dfs:
        merged_df = pd.concat(partial_dfs).drop_duplicates().reset_index(drop=True)
        return merged_df.to_json(orient='records', force_ascii=False, indent=2)

    # ③ 아무 것도 없으면 빈 JSON 반환
    return json.dumps([], ensure_ascii=False)


if __name__ == "__main__":
    # 테스트용 예시 데이터


    # 예시 입력
    input_ingre = ['도토리묵', '상추', '당근', '양파','독']

    print(filter_recipes(input_ingre, df))


[
  {
    "RCP_SNO":2603077,
    "CKG_NM":"도토리묵무침",
    "CKG_MTRL_CN":"[재료] 도토리묵 1모| 상추 5장| 쑥갓 2줄기| 깻잎 4장| 양파 1\/2개| 오이 1\/2개| 당근 1\/3개 [양념] 간장 3큰술| 깨소금 1큰술| 참기름 1큰술| 고춧가루 1큰술| 다진마늘 1\/2큰술| 다진대파 2큰술| 설탕 1큰술",
    "rec1":"['도토리묵', '상추', '쑥갓', '깻잎', '양파', '오이', '당근', '마늘', '다진파']"
  },
  {
    "RCP_SNO":6593198,
    "CKG_NM":"도토리묵무침",
    "CKG_MTRL_CN":"[재료] 도토리묵 1모| 상추(or 깻잎) 4~5장| 양파 약간| 당근 약간 [양념] 간장 4큰술| 고춧가루 2큰술| 올리고당 1~2큰술| 매실액 2큰술| 다진마늘 1\/2큰술| 들기름(or 참기름) 1큰술| 통깨",
    "rec1":"['도토리묵', '상추', '깻잎', '양파', '당근', '마늘']"
  },
  {
    "RCP_SNO":6726120,
    "CKG_NM":"도토리묵무침",
    "CKG_MTRL_CN":"[재료] 도토리묵| 상추| 깻잎| 양파| 오이| 당근| 고추 [양념] 고추가루 1T| 간장 2T| 참기름 0.5T| 다진마늘 1T| 설탕 0.5T",
    "rec1":"['도토리묵', '상추', '깻잎', '양파', '오이', '당근', '고추', '마늘']"
  },
  {
    "RCP_SNO":6830788,
    "CKG_NM":"도토리묵사발",
    "CKG_MTRL_CN":"[도토리묵 쑤기 재료] 도토리가루 1컵| 생수 5+1\/2컵 [묵사발 재료] 도토리 1\/2| 오이| 당근| 양파| 파| 상추| 쑥갓| 배추김치| 방울토마토 [육수 재료] 멸치| 건새우| 통마늘| 대파| 무| 표고버섯| 양파| 다시마 [양념] 소금 1수저| 설탕 1\/2수저| 식초 약간| 김가루| 깨소금",
   